In [ ]:
import marimo as mo

In [ ]:
mo.md(r"""
# GLAD & MACE Ensemble
### GLAD (Generative model of Labels, Abilities, and Difficulties)
- EM based algorithm that uses a logistic function to recover true labels based on varying task difficulty
$$P(L_{ij} = Z_i \mid \alpha_j, \beta_i) = \frac{1}{1 + e^{-\alpha_j \beta_i}}$$
- Each item i has a true label $Z_i$  and a difficulty $1/\beta_i$
- Each annotator j has an ability $\alpha_j$ (more positive, more reliable)
- P is the probability that annotator j labels item i correctly
- if $\alpha_j$ is high, annotator is right regardless of difficulty. If $\beta_i$ is high, most annotators get it right regardless of skill

---

### MACE (Multi-Annotator Competence Estimation)
- Each annotator j has a competence parameter $\theta_j$ = probability that they're in 'diligent' state when labelling items as opposed to 'spamming'
- Algorithm uses bernoulli latent variables

$S_{ij} = \begin{cases}
0 & \text{if annotator } j \text{ knows the true label for instance } i, \\
1 & \text{if annotator } j \text{ spams/guesses on instance } i.
\end{cases}$


""")

<span class="markdown prose dark:prose-invert contents"><h1 id="glad-mace-ensemble">GLAD &amp; MACE Ensemble</h1>
<h3 id="glad-generative-model-of-labels-abilities-and-difficulties">GLAD (Generative model of Labels, Abilities, and Difficulties)</h3>
<ul>
<li>EM based algorithm that uses a logistic function to recover true labels based on varying task difficulty</li>
</ul>
$$P(L_{ij} = Z_i \mid \alpha_j, \beta_i) = \frac{1}{1 + e^{-\alpha_j \beta_i}}$$<ul>
<li>Each item i has a true label $Z_i$  and a difficulty $1/\beta_i$</li>
<li>Each annotator j has an ability $\alpha_j$ (more positive, more reliable)</li>
<li>P is the probability that annotator j labels item i correctly</li>
<li>if $\alpha_j$ is high, annotator is right regardless of difficulty. If $\beta_i$ is high, most annotators get it right regardless of skill</li>
</ul>
<hr />
<h3 id="mace-multi-annotator-competence-estimation">MACE (Multi-Annotator Competence Estimation)</h3>
<ul>
<li>Each annotator j has a competence parameter $\theta_j$ = probability that they're in 'diligent' state when labelling items as opposed to 'spamming'</li>
<li>Algorithm uses bernoulli latent variables</li>
</ul>
<span class="paragraph">$S_{ij} = \begin{cases}
0 &amp; \text{if annotator } j \text{ knows the true label for instance } i, \\
1 &amp; \text{if annotator } j \text{ spams/guesses on instance } i.
\end{cases}$</span></span>

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit as sigmoid, logsumexp
from collections import defaultdict, Counter
from typing import List, Any

In [ ]:
def softplus(x):
    """
    stable logistic function, constraints beta to positive
    """
    return np.logaddexp(0.0, x)

In [ ]:
mo.md(r"""
# Inner ensemble engine
""")

<span class="markdown prose dark:prose-invert contents"><h1 id="inner-ensemble-engine">Inner ensemble engine</h1></span>

In [ ]:
class GladMaceHybridModel:

    def __init__(self, number_of_classes, ability_regularization=0.01,
                 ease_regularization=0.01, diligence_regularization=0.01,
                 spam_distribution_smoothing=1.0, random_seed=0):
        
        self.number_of_classes = number_of_classes
        self.ability_regularization = ability_regularization #alpha
        self.ease_regularization = ease_regularization #beta
        self.diligence_regularization = diligence_regularization #theta
        self.spam_distribution_smoothing = spam_distribution_smoothing #dirichlet
        self.random_number_generator = np.random.default_rng(random_seed)

        #will be populated when .fit is called
        self.annotator_ability = None                 #(num_annotators,)
        self.item_ease = None                          #(num_items,)
        self.annotator_diligence_tendency = None        #(num_annotators,)
        self.annotator_spam_label_distribution = None   #(num_annotators, num_classes)
        self.log_class_prior = None                     #(num_classes,)
        self.item_ids = None
        self.annotator_ids = None

    def _prepare_annotation_arrays(self, annotation_triples):
        """
        converts iterable triples into int index arrays to work with
        """
        annotation_triples = list(annotation_triples)
        item_ids = sorted({triple[0] for triple in annotation_triples})
        annotator_ids = sorted({triple[1] for triple in annotation_triples})
        item_id_to_index = {item_id: index for index, item_id in enumerate(item_ids)}
        annotator_id_to_index = {annotator_id: index for index, annotator_id in enumerate(annotator_ids)}

        item_index_per_annotation = np.array(
            [item_id_to_index[triple[0]] for triple in annotation_triples], dtype=np.int64
        )
        annotator_index_per_annotation = np.array(
            [annotator_id_to_index[triple[1]] for triple in annotation_triples], dtype=np.int64
        )
        label_per_annotation = np.array(
            [int(triple[2]) for triple in annotation_triples], dtype=np.int64
        )

        annotation_data = {
            "item_index_per_annotation": item_index_per_annotation,
            "annotator_index_per_annotation": annotator_index_per_annotation,
            "label_per_annotation": label_per_annotation,
        }
        return item_ids, annotator_ids, annotation_data, len(item_ids), len(annotator_ids)

    def _compute_log_joint_and_log_marginal(self, annotation_data, number_of_items,
                                              annotator_ability, item_ease,
                                              annotator_diligence_tendency,
                                              annotator_spam_label_distribution,
                                              log_class_prior):
        """
        calculates log prob of label being correct for item with the log joint, vectorised
        """
        number_of_classes = self.number_of_classes
        item_index_per_annotation = annotation_data["item_index_per_annotation"]
        annotator_index_per_annotation = annotation_data["annotator_index_per_annotation"]
        label_per_annotation = annotation_data["label_per_annotation"]

        #proba this annotation was diligent
        diligence_probability = sigmoid(
            annotator_diligence_tendency[annotator_index_per_annotation]
            + item_ease[item_index_per_annotation]
        )
        #proba annot's label is correct if diligent
        correct_probability_if_diligent = sigmoid(
            annotator_ability[annotator_index_per_annotation] * item_ease[item_index_per_annotation]
        )
        #wrong label spread evenly assumption if diligent but wrong over classes
        incorrect_probability_if_diligent = (
            (1.0 - correct_probability_if_diligent) / max(number_of_classes - 1, 1)
        )
        #proba of observed label under annot's personal spam labeling habit
        probability_of_label_if_spamming = annotator_spam_label_distribution[
            annotator_index_per_annotation, label_per_annotation
        ]

        log_joint_probability = np.tile(log_class_prior, (number_of_items, 1)).astype(float)
        for candidate_label in range(number_of_classes):
            annotation_matches_candidate = (label_per_annotation == candidate_label)
            probability_correct_given_candidate = np.where(
                annotation_matches_candidate,
                correct_probability_if_diligent,
                incorrect_probability_if_diligent,
            )
            probability_of_this_annotation_given_candidate = np.clip(
                diligence_probability * probability_correct_given_candidate
                + (1.0 - diligence_probability) * probability_of_label_if_spamming,
                1e-12, None,
            )
            log_probability_of_this_annotation = np.log(probability_of_this_annotation_given_candidate)

            summed_log_probability_per_item = np.zeros(number_of_items)
            np.add.at(summed_log_probability_per_item, item_index_per_annotation, log_probability_of_this_annotation)
            log_joint_probability[:, candidate_label] += summed_log_probability_per_item

        log_marginal_probability = logsumexp(log_joint_probability, axis=1)
        return log_joint_probability, log_marginal_probability

    def _negative_log_likelihood_and_gradient(self, parameter_vector, annotation_data,
                                                number_of_items, number_of_annotators,
                                                annotator_spam_label_distribution, log_class_prior):
        """
        calculates neg log likelihood of observed data + regularisation penalty, exact gradient over approximation with finite difference since better speed 
        """
        number_of_classes = self.number_of_classes
        number_of_annotators_ = number_of_annotators
        number_of_items_ = number_of_items

        annotator_ability = parameter_vector[:number_of_annotators_]
        raw_item_ease = parameter_vector[number_of_annotators_:number_of_annotators_ + number_of_items_]
        item_ease = softplus(raw_item_ease)
        annotator_diligence_tendency = parameter_vector[
            number_of_annotators_ + number_of_items_: number_of_annotators_ + number_of_items_ + number_of_annotators_
        ]

        item_index_per_annotation = annotation_data["item_index_per_annotation"]
        annotator_index_per_annotation = annotation_data["annotator_index_per_annotation"]
        label_per_annotation = annotation_data["label_per_annotation"]
        number_of_annotations = len(label_per_annotation)

        diligence_probability = sigmoid(
            annotator_diligence_tendency[annotator_index_per_annotation] + item_ease[item_index_per_annotation]
        )
        correct_probability_if_diligent = sigmoid(
            annotator_ability[annotator_index_per_annotation] * item_ease[item_index_per_annotation]
        )
        incorrect_probability_if_diligent = (
            (1.0 - correct_probability_if_diligent) / max(number_of_classes - 1, 1)
        )
        probability_of_label_if_spamming = annotator_spam_label_distribution[
            annotator_index_per_annotation, label_per_annotation
        ]

        log_joint_probability = np.tile(log_class_prior, (number_of_items_, 1)).astype(float)
        probability_of_annotation_per_candidate = np.zeros((number_of_annotations, number_of_classes))
        probability_correct_given_candidate_per_candidate = np.zeros((number_of_annotations, number_of_classes))
        for candidate_label in range(number_of_classes):
            annotation_matches_candidate = (label_per_annotation == candidate_label)
            probability_correct_given_candidate = np.where(
                annotation_matches_candidate, correct_probability_if_diligent, incorrect_probability_if_diligent
            )
            probability_of_annotation = np.clip(
                diligence_probability * probability_correct_given_candidate
                + (1.0 - diligence_probability) * probability_of_label_if_spamming,
                1e-12, None,
            )
            probability_of_annotation_per_candidate[:, candidate_label] = probability_of_annotation
            probability_correct_given_candidate_per_candidate[:, candidate_label] = probability_correct_given_candidate

            summed_log_probability_per_item = np.zeros(number_of_items_)
            np.add.at(summed_log_probability_per_item, item_index_per_annotation, np.log(probability_of_annotation))
            log_joint_probability[:, candidate_label] += summed_log_probability_per_item

        log_marginal_probability = logsumexp(log_joint_probability, axis=1)
        negative_log_likelihood = -np.sum(log_marginal_probability)

        #posterior proba over true label for each item for weighting gradient contributions
        true_label_posterior = np.exp(log_joint_probability - log_marginal_probability[:, None])

        gradient_wrt_diligence_probability = np.zeros(number_of_annotations)
        gradient_wrt_correct_probability = np.zeros(number_of_annotations)
        for candidate_label in range(number_of_classes):
            posterior_weight_for_candidate = true_label_posterior[item_index_per_annotation, candidate_label]
            gradient_wrt_probability_of_annotation = (
                -posterior_weight_for_candidate / probability_of_annotation_per_candidate[:, candidate_label]
            )
            gradient_wrt_diligence_probability += gradient_wrt_probability_of_annotation * (
                probability_correct_given_candidate_per_candidate[:, candidate_label]
                - probability_of_label_if_spamming
            )
            annotation_matches_candidate = (label_per_annotation == candidate_label)
            derivative_of_correct_given_candidate_wrt_correct_probability = np.where(
                annotation_matches_candidate, 1.0, -1.0 / max(number_of_classes - 1, 1)
            )
            gradient_wrt_correct_probability += (
                gradient_wrt_probability_of_annotation
                * diligence_probability
                * derivative_of_correct_given_candidate_wrt_correct_probability
            )

        #chain rule for raw parameters via sigmoid derivative
        derivative_of_diligence_wrt_diligence_tendency_or_ease = diligence_probability * (1.0 - diligence_probability)
        derivative_of_correct_probability_wrt_ability = (
            correct_probability_if_diligent * (1.0 - correct_probability_if_diligent)
            * item_ease[item_index_per_annotation]
        )
        derivative_of_correct_probability_wrt_ease = (
            correct_probability_if_diligent * (1.0 - correct_probability_if_diligent)
            * annotator_ability[annotator_index_per_annotation]
        )

        contribution_to_diligence_tendency_gradient = (
            gradient_wrt_diligence_probability * derivative_of_diligence_wrt_diligence_tendency_or_ease
        )
        contribution_to_ability_gradient = (
            gradient_wrt_correct_probability * derivative_of_correct_probability_wrt_ability
        )
        contribution_to_ease_gradient = (
            gradient_wrt_diligence_probability * derivative_of_diligence_wrt_diligence_tendency_or_ease
            + gradient_wrt_correct_probability * derivative_of_correct_probability_wrt_ease
        )

        gradient_wrt_ability = np.zeros(number_of_annotators_)
        gradient_wrt_ease = np.zeros(number_of_items_)
        gradient_wrt_diligence_tendency = np.zeros(number_of_annotators_)
        np.add.at(gradient_wrt_diligence_tendency, annotator_index_per_annotation, contribution_to_diligence_tendency_gradient)
        np.add.at(gradient_wrt_ability, annotator_index_per_annotation, contribution_to_ability_gradient)
        np.add.at(gradient_wrt_ease, item_index_per_annotation, contribution_to_ease_gradient)

        regularization_penalty = (
            self.ability_regularization * np.sum(annotator_ability ** 2)
            + self.ease_regularization * np.sum(item_ease ** 2)
            + self.diligence_regularization * np.sum(annotator_diligence_tendency ** 2)
        )
        gradient_wrt_ability += 2 * self.ability_regularization * annotator_ability
        gradient_wrt_ease += 2 * self.ease_regularization * item_ease
        gradient_wrt_diligence_tendency += 2 * self.diligence_regularization * annotator_diligence_tendency

        #chain thru softplus transform
        gradient_wrt_raw_item_ease = gradient_wrt_ease * sigmoid(raw_item_ease)

        gradient_vector = np.concatenate(
            [gradient_wrt_ability, gradient_wrt_raw_item_ease, gradient_wrt_diligence_tendency]
        )
        objective_value = negative_log_likelihood + regularization_penalty
        return objective_value, gradient_vector

    def _negative_log_likelihood_only(self, parameter_vector, annotation_data, number_of_items,
                                        number_of_annotators, annotator_spam_label_distribution, log_class_prior):
        """same obj as previous function without gradient, used for progress reporting"""
        objective_value, _ = self._negative_log_likelihood_and_gradient(
            parameter_vector, annotation_data, number_of_items, number_of_annotators,
            annotator_spam_label_distribution, log_class_prior,
        )
        return objective_value

    def _update_spam_label_distribution(self, annotation_data, number_of_items, number_of_annotators,
                                          annotator_ability, item_ease, annotator_diligence_tendency,
                                          log_class_prior):
        """
        re estimates each annotator's spam label distribution/habit given current true label posteriors for every item, has dirichlet smoothed weighted avg
        """
        number_of_classes = self.number_of_classes
        item_index_per_annotation = annotation_data["item_index_per_annotation"]
        annotator_index_per_annotation = annotation_data["annotator_index_per_annotation"]
        label_per_annotation = annotation_data["label_per_annotation"]

        log_joint_probability, log_marginal_probability = self._compute_log_joint_and_log_marginal(
            annotation_data, number_of_items, annotator_ability, item_ease,
            annotator_diligence_tendency, self.annotator_spam_label_distribution, log_class_prior,
        )
        true_label_posterior = np.exp(log_joint_probability - log_marginal_probability[:, None])

        diligence_probability = sigmoid(
            annotator_diligence_tendency[annotator_index_per_annotation] + item_ease[item_index_per_annotation]
        )
        correct_probability_if_diligent = sigmoid(
            annotator_ability[annotator_index_per_annotation] * item_ease[item_index_per_annotation]
        )
        incorrect_probability_if_diligent = (
            (1.0 - correct_probability_if_diligent) / max(number_of_classes - 1, 1)
        )
        probability_of_label_if_spamming = self.annotator_spam_label_distribution[
            annotator_index_per_annotation, label_per_annotation
        ]

        expected_spam_responsibility_per_annotation = np.zeros(len(label_per_annotation))
        for candidate_label in range(number_of_classes):
            annotation_matches_candidate = (label_per_annotation == candidate_label)
            probability_correct_given_candidate = np.where(
                annotation_matches_candidate, correct_probability_if_diligent, incorrect_probability_if_diligent
            )
            probability_of_annotation_given_candidate = np.clip(
                diligence_probability * probability_correct_given_candidate
                + (1.0 - diligence_probability) * probability_of_label_if_spamming,
                1e-12, None,
            )
            probability_spam_given_candidate = (
                (1.0 - diligence_probability) * probability_of_label_if_spamming
                / probability_of_annotation_given_candidate
            )
            posterior_weight_for_candidate = true_label_posterior[item_index_per_annotation, candidate_label]
            expected_spam_responsibility_per_annotation += posterior_weight_for_candidate * probability_spam_given_candidate

        pseudo_counts = np.full(
            (number_of_annotators, number_of_classes),
            self.spam_distribution_smoothing / number_of_classes,
        )
        np.add.at(
            pseudo_counts,
            (annotator_index_per_annotation, label_per_annotation),
            expected_spam_responsibility_per_annotation,
        )
        self.annotator_spam_label_distribution = pseudo_counts / pseudo_counts.sum(axis=1, keepdims=True)

    def fit(self, annotation_triples, max_outer_iterations=15, max_optimizer_iterations=100,
            initial_annotator_ability=None, initial_annotator_diligence_tendency=None,
            verbose=True, convergence_tolerance=1e-4):
        
        (self.item_ids, self.annotator_ids, annotation_data,
         number_of_items, number_of_annotators) = self._prepare_annotation_arrays(annotation_triples)
        number_of_classes = self.number_of_classes

        annotator_ability = np.zeros(number_of_annotators)
        raw_item_ease = np.zeros(number_of_items)  #softplus(0) is nice starting ease
        annotator_diligence_tendency = np.zeros(number_of_annotators)

        if initial_annotator_ability:
            for annotator_id, value in initial_annotator_ability.items():
                if annotator_id in self.annotator_ids:
                    annotator_ability[self.annotator_ids.index(annotator_id)] = value
        if initial_annotator_diligence_tendency:
            for annotator_id, value in initial_annotator_diligence_tendency.items():
                if annotator_id in self.annotator_ids:
                    annotator_diligence_tendency[self.annotator_ids.index(annotator_id)] = value

        annotator_ability += self.random_number_generator.normal(0, 0.1, size=number_of_annotators)
        raw_item_ease += self.random_number_generator.normal(0, 0.1, size=number_of_items)

        self.annotator_spam_label_distribution = np.full(
            (number_of_annotators, number_of_classes), 1.0 / number_of_classes
        )
        #if you dont have prior reason to expect one class to be more common than another, might as well use fixed uniform prior over classes
        self.log_class_prior = np.full(number_of_classes, -np.log(number_of_classes))

        previous_negative_log_likelihood = np.inf
        for outer_iteration in range(max_outer_iterations):
            #first optimise continuous parameters e.b. ability, ease, diligence thru gradient based optimisation
            initial_parameter_vector = np.concatenate([annotator_ability, raw_item_ease, annotator_diligence_tendency])
            optimization_result = minimize(
                self._negative_log_likelihood_and_gradient,
                initial_parameter_vector,
                args=(annotation_data, number_of_items, number_of_annotators,
                      self.annotator_spam_label_distribution, self.log_class_prior),
                method="L-BFGS-B", jac=True, options={"maxiter": max_optimizer_iterations},
            )
            annotator_ability = optimization_result.x[:number_of_annotators]
            raw_item_ease = optimization_result.x[number_of_annotators:number_of_annotators + number_of_items]
            annotator_diligence_tendency = optimization_result.x[
                number_of_annotators + number_of_items: number_of_annotators + number_of_items + number_of_annotators
            ]
            item_ease = softplus(raw_item_ease)

            #then update spam distribution while holding continuous parameters fixed
            self._update_spam_label_distribution(
                annotation_data, number_of_items, number_of_annotators,
                annotator_ability, item_ease, annotator_diligence_tendency, self.log_class_prior,
            )

            current_negative_log_likelihood = self._negative_log_likelihood_only(
                np.concatenate([annotator_ability, raw_item_ease, annotator_diligence_tendency]),
                annotation_data, number_of_items, number_of_annotators,
                self.annotator_spam_label_distribution, self.log_class_prior,
            )
            if verbose:
                print(f"[round {outer_iteration:02d}] negative log-likelihood = {current_negative_log_likelihood:.3f}")

            relative_change = abs(previous_negative_log_likelihood - current_negative_log_likelihood)
            if relative_change < convergence_tolerance * max(1.0, abs(previous_negative_log_likelihood)):
                if verbose:
                    print("Converged.")
                break
            previous_negative_log_likelihood = current_negative_log_likelihood

        self.annotator_ability = annotator_ability
        self.item_ease = item_ease
        self.annotator_diligence_tendency = annotator_diligence_tendency
        self._annotation_data_from_fit = annotation_data
        self._number_of_items_from_fit = number_of_items
        self._cached_label_distributions = self.predict_label_distributions()
        return self

    def predict_label_distributions(self, annotation_triples=None):
        """
        return model's belief about true label of each item via probability distrbution
        """
        if annotation_triples is None:
            annotation_data = self._annotation_data_from_fit
            number_of_items = self._number_of_items_from_fit
        else:
            _, _, annotation_data, number_of_items, _ = self._prepare_annotation_arrays(annotation_triples)

        log_joint_probability, log_marginal_probability = self._compute_log_joint_and_log_marginal(
            annotation_data, number_of_items, self.annotator_ability, self.item_ease,
            self.annotator_diligence_tendency, self.annotator_spam_label_distribution, self.log_class_prior,
        )
        true_label_posterior = np.exp(log_joint_probability - log_marginal_probability[:, None])
        return {self.item_ids[index]: true_label_posterior[index] for index in range(number_of_items)}

    def describe_items(self):
        """
        returns diagnostics per item
        """
        label_distributions = self._cached_label_distributions
        item_report = {}
        for item_id, distribution in label_distributions.items():
            entropy = -np.sum(distribution * np.log(np.clip(distribution, 1e-12, None)))
            item_index = self.item_ids.index(item_id)
            item_report[item_id] = {
                "label_uncertainty": float(entropy),
                "ease": float(self.item_ease[item_index]),
            }
        return item_report

    def describe_annotators(self):
        """
        returns diagnostics per annotator
        """
        annotator_report = {}
        average_item_ease = float(np.mean(self.item_ease))
        for annotator_index, annotator_id in enumerate(self.annotator_ids):
            annotator_report[annotator_id] = {
                "estimated_ability": float(self.annotator_ability[annotator_index]),
                "estimated_diligence_on_a_typical_item": float(
                    sigmoid(self.annotator_diligence_tendency[annotator_index] + average_item_ease)
                ),
                "personal_spam_label_habits": self.annotator_spam_label_distribution[annotator_index].copy(),
            }
        return annotator_report

In [ ]:
mo.md(r"""
# fake crowdsourcing dataset for testing
""")

<span class="markdown prose dark:prose-invert contents"><h1 id="fake-crowdsourcing-dataset-for-testing">fake crowdsourcing dataset for testing</h1></span>

In [ ]:
random_number_generator = np.random.default_rng(42)

NUMBER_OF_CLASSES = 4
NUMBER_OF_ITEMS = 200
NUMBER_OF_ANNOTATORS = 15

#ground truth that model will never see
true_label_per_item = random_number_generator.integers(0, NUMBER_OF_CLASSES, size=NUMBER_OF_ITEMS)

#first half is ok second is hard
true_ease_per_item = np.concatenate([
    np.full(NUMBER_OF_ITEMS // 2, 2.0),
    np.full(NUMBER_OF_ITEMS - NUMBER_OF_ITEMS // 2, 0.1),
])


'''annotator roles:
0-4 = skilled/diligent
5-9 = diligent with mediocre skill
10-12 = spammers w a systematic bias where they favor class 0 when spamming
13-14 = spammers who closely guess uniformly at random
'''
true_ability_per_annotator = np.zeros(NUMBER_OF_ANNOTATORS)
true_diligence_tendency_per_annotator = np.zeros(NUMBER_OF_ANNOTATORS)
true_spam_label_distribution_per_annotator = np.full(
    (NUMBER_OF_ANNOTATORS, NUMBER_OF_CLASSES), 1.0 / NUMBER_OF_CLASSES
)

true_ability_per_annotator[0:5] = random_number_generator.uniform(1.5, 2.5, 5)
true_diligence_tendency_per_annotator[0:5] = 3.0  # almost always diligent

true_ability_per_annotator[5:10] = random_number_generator.uniform(0.1, 0.5, 5)
true_diligence_tendency_per_annotator[5:10] = 3.0

true_diligence_tendency_per_annotator[10:13] = -3.0  # almost always spamming
biased_spam_distribution = np.full(NUMBER_OF_CLASSES, 0.05)
biased_spam_distribution[0] = 1.0 - 0.05 * (NUMBER_OF_CLASSES - 1)
true_spam_label_distribution_per_annotator[10:13] = biased_spam_distribution

true_diligence_tendency_per_annotator[13:15] = -3.0
#true_spam_label_distribution_per_annotator is already uniform for these 2

annotation_triples = []
for item_index in range(NUMBER_OF_ITEMS):
    annotators_for_this_item = random_number_generator.choice(NUMBER_OF_ANNOTATORS, size=6, replace=False)
    for annotator_index in annotators_for_this_item:
        diligence_probability = sigmoid(
            true_diligence_tendency_per_annotator[annotator_index] + true_ease_per_item[item_index]
        )
        annotator_is_diligent_this_time = random_number_generator.random() < diligence_probability

        if annotator_is_diligent_this_time:
            correct_probability = sigmoid(
                true_ability_per_annotator[annotator_index] * true_ease_per_item[item_index]
            )
            if random_number_generator.random() < correct_probability:
                observed_label = true_label_per_item[item_index]
            else:
                other_classes = [c for c in range(NUMBER_OF_CLASSES) if c != true_label_per_item[item_index]]
                observed_label = random_number_generator.choice(other_classes)
        else:
            observed_label = random_number_generator.choice(
                NUMBER_OF_CLASSES, p=true_spam_label_distribution_per_annotator[annotator_index]
            )

        annotation_triples.append((item_index, annotator_index, observed_label))

print(f"Total annotations: {len(annotation_triples)}")

#fitting model but it only ever seens annotation_triples
model = GladMaceHybridModel(number_of_classes=NUMBER_OF_CLASSES, random_seed=1)
model.fit(annotation_triples, max_outer_iterations=12, max_optimizer_iterations=150, verbose=True)

label_distributions = model.predict_label_distributions()
predicted_label_per_item = np.array([np.argmax(label_distributions[i]) for i in range(NUMBER_OF_ITEMS)])
accuracy = np.mean(predicted_label_per_item == true_label_per_item)
print(f"\nHard-label accuracy vs true labels: {accuracy:.3f}")

#checking if annotator roles were recovered relatively ok
annotator_report = model.describe_annotators()
print("\nAnnotator diagnostics (estimated ability, estimated diligence on a typical item):")
for annotator_index in range(NUMBER_OF_ANNOTATORS):
    diagnostics = annotator_report[annotator_index]
    if annotator_index < 5:
        true_role = "skilled & diligent"
    elif annotator_index < 10:
        true_role = "mediocre & diligent"
    elif annotator_index < 13:
        true_role = "biased spammer"
    else:
        true_role = "random spammer"
    print(
        f"  annotator {annotator_index:2d} [{true_role:20s}] "
        f"ability={diagnostics['estimated_ability']:+.2f}  "
        f"diligence={diagnostics['estimated_diligence_on_a_typical_item']:.2f}"
    )

#check if item dificulty shows up as expected in label uncertainy
item_report = model.describe_items()
mean_uncertainty_on_easy_items = np.mean(
    [item_report[i]["label_uncertainty"] for i in range(NUMBER_OF_ITEMS // 2)]
)
mean_uncertainty_on_hard_items = np.mean(
    [item_report[i]["label_uncertainty"] for i in range(NUMBER_OF_ITEMS // 2, NUMBER_OF_ITEMS)]
)
print(f"\nMean label uncertainty on EASY items: {mean_uncertainty_on_easy_items:.3f}")
print(f"Mean label uncertainty on HARD items: {mean_uncertainty_on_hard_items:.3f}")


#show soft GT output for a few items since its the main point of the model to show uncertainty of adversarial items without collpsing into one wrong label
uncertainty_per_item = np.array([item_report[i]["label_uncertainty"] for i in range(NUMBER_OF_ITEMS)])
most_contested_items = np.argsort(-uncertainty_per_item)[:3]
most_confident_items = np.argsort(uncertainty_per_item)[:3]

print("\nMost CONTESTED items (soft distribution kept, not collapsed to one label):")
for item_index in most_contested_items:
    distribution_string = ", ".join(
        f"class{k}={p:.2f}" for k, p in enumerate(label_distributions[item_index])
    )
    print(
        f"  item {item_index:3d} (true={true_label_per_item[item_index]}, "
        f"ease={item_report[item_index]['ease']:.2f}): {distribution_string}"
    )

print("\nMost CONFIDENT items:")
for item_index in most_confident_items:
    distribution_string = ", ".join(
        f"class{k}={p:.2f}" for k, p in enumerate(label_distributions[item_index])
    )
    print(
        f"  item {item_index:3d} (true={true_label_per_item[item_index]}, "
        f"ease={item_report[item_index]['ease']:.2f}): {distribution_string}"
    )

#compare against majority vote for baseline sanity check
labels_by_item = {}
for item_index, annotator_index, observed_label in annotation_triples:
    labels_by_item.setdefault(item_index, []).append(observed_label)

majority_vote_prediction = [
    Counter(labels_by_item[item_index]).most_common(1)[0][0] for item_index in range(NUMBER_OF_ITEMS)
]
majority_vote_accuracy = np.mean(np.array(majority_vote_prediction) == true_label_per_item)
print(f"\nMajority-vote baseline accuracy: {majority_vote_accuracy:.3f}")
print(f"Hybrid model accuracy:           {accuracy:.3f}")

Total annotations: 1200
[round 00] negative log-likelihood = 1334.981
[round 01] negative log-likelihood = 1203.284
[round 02] negative log-likelihood = 1196.909
[round 03] negative log-likelihood = 1195.941
[round 04] negative log-likelihood = 1195.231
[round 05] negative log-likelihood = 1194.727
[round 06] negative log-likelihood = 1194.416
[round 07] negative log-likelihood = 1194.219
[round 08] negative log-likelihood = 1194.079
[round 09] negative log-likelihood = 1193.961
Converged.

Hard-label accuracy vs true labels: 0.820

Annotator diagnostics (estimated ability, estimated diligence on a typical item):
  annotator  0 [skilled & diligent  ] ability=+8.98  diligence=0.99
  annotator  1 [skilled & diligent  ] ability=+8.40  diligence=1.00
  annotator  2 [skilled & diligent  ] ability=+7.66  diligence=1.00
  annotator  3 [skilled & diligent  ] ability=+7.49  diligence=1.00
  annotator  4 [skilled & diligent  ] ability=+9.81  diligence=1.00
  annotator  5 [mediocre & diligent ] a